<a href="https://colab.research.google.com/github/TienPhap0102/06_SmartToys-Ecommerce-Profitability-Customer-Journey-Analytics/blob/main/02_build_mart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Kéo dữ liệu staging về

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "jda-k1"
dataset_id = "bigproject_phap_staging"
key_path = "/content/drive/MyDrive/1. Data anh Huy/Big Project Ecommerce/Dataset/jda-k1-2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(
    key_path
)

client = bigquery.Client(
    credentials=credentials,
    project=project_id
)

In [ ]:
%pip install -q db-dtypes

table_names = [
    "products",
    "website_sessions",
    "website_pageviews",
    "orders",
    "order_items",
    "order_item_refunds"
]

staging_tables = {}

for table_name in table_names:
    query = f"""
    SELECT *
    FROM `jda-k1.bigproject_phap_staging.{table_name}`
    """

    staging_tables[table_name] = (
        client.query(query)
        .to_dataframe(create_bqstorage_client=False)
    )

    print(f"{table_name}: {len(staging_tables[table_name]):,} dòng")

products: 4 dòng
website_sessions: 472,871 dòng
website_pageviews: 1,188,124 dòng
orders: 32,313 dòng
order_items: 40,025 dòng
order_item_refunds: 1,731 dòng


In [ ]:
staging_tables['products'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    4 non-null      Int64         
 1   created_at    4 non-null      datetime64[us]
 2   product_name  4 non-null      object        
dtypes: Int64(1), datetime64[us](1), object(1)
memory usage: 232.0+ bytes


In [ ]:
staging_tables['products']

,product_id,created_at,product_name
0,1,2012-03-19 08:00:00,The Original Mr. Fuzzy
1,2,2013-01-06 13:00:00,The Forever Love Bear
2,3,2013-12-12 09:00:00,The Birthday Sugar Panda
3,4,2014-02-05 10:00:00,The Hudson River Mini bear


In [ ]:
staging_tables['orders']

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49
...,...,...,...,...,...,...,...,...
32308,28352,2015-01-22 19:14:41,426655,328436,4,2,89.98,31.98
32309,28395,2015-01-23 09:23:45,427024,358361,4,2,89.98,31.98
32310,30807,2015-02-25 07:12:43,454593,379896,4,2,89.98,31.98
32311,31219,2015-03-03 15:28:08,459794,384020,4,2,89.98,31.98


In [ ]:
fact_orders = staging_tables['orders'].copy()
fact_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            32313 non-null  Int64         
 1   created_at          32313 non-null  datetime64[us]
 2   website_session_id  32313 non-null  Int64         
 3   user_id             32313 non-null  Int64         
 4   primary_product_id  32313 non-null  Int64         
 5   items_purchased     32313 non-null  Int64         
 6   price_usd           32313 non-null  float64       
 7   cogs_usd            32313 non-null  float64       
dtypes: Int64(5), datetime64[us](1), float64(2)
memory usage: 2.1 MB


In [ ]:
staging_tables['order_items']

,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,11466,2014-02-05 11:02:39,11148,4,0,29.99,9.49
1,11468,2014-02-05 12:15:56,11149,4,0,29.99,9.49
2,11470,2014-02-05 13:07:28,11150,4,0,29.99,9.49
3,11473,2014-02-05 13:32:46,11152,4,0,29.99,9.49
4,11475,2014-02-05 13:45:17,11153,4,0,29.99,9.49
...,...,...,...,...,...,...,...
40020,39987,2015-03-18 17:03:05,32283,2,1,59.99,22.49
40021,39990,2015-03-18 17:38:05,32286,2,1,59.99,22.49
40022,39992,2015-03-18 18:10:53,32288,2,1,59.99,22.49
40023,40013,2015-03-19 00:19:57,32303,2,1,59.99,22.49


In [ ]:
fact_order_items = staging_tables['order_items'].copy()
fact_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40025 entries, 0 to 40024
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_item_id    40025 non-null  Int64         
 1   created_at       40025 non-null  datetime64[us]
 2   order_id         40025 non-null  Int64         
 3   product_id       40025 non-null  Int64         
 4   is_primary_item  40025 non-null  Int64         
 5   price_usd        40025 non-null  float64       
 6   cogs_usd         40025 non-null  float64       
dtypes: Int64(4), datetime64[us](1), float64(2)
memory usage: 2.3 MB


In [ ]:
fact_refunds = staging_tables['order_item_refunds'].copy()
fact_refunds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1731 entries, 0 to 1730
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_item_refund_id  1731 non-null   Int64         
 1   created_at            1731 non-null   datetime64[us]
 2   order_item_id         1731 non-null   Int64         
 3   order_id              1731 non-null   Int64         
 4   refund_amount_usd     1731 non-null   float64       
dtypes: Int64(3), datetime64[us](1), float64(1)
memory usage: 72.8 KB


In [ ]:
fact_sessions = staging_tables['website_sessions'].copy()
fact_sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   website_session_id  472871 non-null  Int64         
 1   created_at          472871 non-null  datetime64[us]
 2   user_id             472871 non-null  Int64         
 3   is_repeat_session   472871 non-null  Int64         
 4   utm_source          389543 non-null  object        
 5   utm_campaign        389543 non-null  object        
 6   utm_content         389543 non-null  object        
 7   device_type         472871 non-null  object        
 8   http_referer        432954 non-null  object        
 9   traffic_source      472871 non-null  object        
 10  source_basis        472871 non-null  object        
dtypes: Int64(3), datetime64[us](1), object(7)
memory usage: 41.0+ MB


In [ ]:
fact_pageviews = staging_tables['website_pageviews'].copy()
fact_pageviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1188124 entries, 0 to 1188123
Data columns (total 4 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   website_pageview_id  1188124 non-null  Int64         
 1   created_at           1188124 non-null  datetime64[us]
 2   website_session_id   1188124 non-null  Int64         
 3   pageview_url         1188124 non-null  object        
dtypes: Int64(2), datetime64[us](1), object(1)
memory usage: 38.5+ MB


In [ ]:
dim_products = staging_tables['products'].copy()
dim_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    4 non-null      Int64         
 1   created_at    4 non-null      datetime64[us]
 2   product_name  4 non-null      object        
dtypes: Int64(1), datetime64[us](1), object(1)
memory usage: 232.0+ bytes


In [ ]:
import pandas as pd

# Lấy phạm vi thời gian từ 6 bảng đã sạch
min_dates = []
max_dates = []

for table_name, df in staging_tables.items():
    # Ensure 'created_at' is datetime and handle potential errors
    dates = pd.to_datetime(
        df["created_at"],
        errors="raise"
    ).dropna()

    if not dates.empty:
        min_dates.append(dates.dt.date.min())
        max_dates.append(dates.dt.date.max())

if not min_dates:
    raise ValueError("Không có ngày hợp lệ để tạo dim_date.")

# Tạo lịch đầy đủ từ đầu năm nhỏ nhất đến cuối năm lớn nhất
start_date = f"{min(min_dates).year}-01-01"
end_date = f"{max(max_dates).year}-12-31"

dim_date = pd.DataFrame({
    "date": pd.date_range(start=start_date, end=end_date, freq="D")
})

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["day"] = dim_date["date"].dt.day

# Adjust day_of_week: Monday = 2, Sunday = 1
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek.apply(lambda x: 1 if x == 6 else x + 2)

# Add new columns as requested
dim_date['is_weekend'] = (dim_date['day_of_week'] == 1) | (dim_date['day_of_week'] == 7)
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['month_name_2'] = dim_date['date'].dt.strftime('%b')
dim_date['quarter_name'] = dim_date['quarter'].apply(lambda x: f'Q{x}')

# Chỉ giữ ngày, không có giờ
dim_date["date"] = dim_date["date"].dt.date

print(f"dim_date: {len(dim_date):,} ngày")
print(f"Từ {start_date} đến {end_date}")

display(dim_date.head())

dim_date: 1,461 ngày
Từ 2012-01-01 đến 2015-12-31


,date,year,quarter,month,year_month,day,day_of_week,is_weekend,month_name,month_name_2,quarter_name
0,2012-01-01,2012,1,1,2012-01,1,1,True,January,Jan,Q1
1,2012-01-02,2012,1,1,2012-01,2,2,False,January,Jan,Q1
2,2012-01-03,2012,1,1,2012-01,3,3,False,January,Jan,Q1
3,2012-01-04,2012,1,1,2012-01,4,4,False,January,Jan,Q1
4,2012-01-05,2012,1,1,2012-01,5,5,False,January,Jan,Q1


In [ ]:
from google.cloud import bigquery

dataset_ref = "jda-k1.bigproject_phap_mart"
table_ref = f"{dataset_ref}.dim_date"

dataset = client.get_dataset(dataset_ref)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("year", "INTEGER"),
        bigquery.SchemaField("quarter", "INTEGER"),
        bigquery.SchemaField("month", "INTEGER"),
        bigquery.SchemaField("year_month", "STRING"),
        bigquery.SchemaField("day", "INTEGER"),
        bigquery.SchemaField("day_of_week", "INTEGER"),
        bigquery.SchemaField("is_weekend", "BOOL"),
        bigquery.SchemaField("month_name", "STRING"),
        bigquery.SchemaField("month_name_2", "STRING"),
        bigquery.SchemaField("quarter_name", "STRING"),
    ]
)

job = client.load_table_from_dataframe(
    dim_date,
    table_ref,
    job_config=job_config,
    location=dataset.location
)
job.result()

loaded_table = client.get_table(table_ref)

if loaded_table.num_rows != len(dim_date):
    raise ValueError("Số dòng trên BigQuery không khớp dim_date.")

print(f"Đã nạp {loaded_table.num_rows:,} dòng vào {table_ref}")

Đã nạp 1,461 dòng vào jda-k1.bigproject_phap_mart.dim_date


# Đẩy fact, dim tables lên Big Query

In [ ]:
fact_dim_tables = {
    "dim_products": dim_products,
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
    "fact_refunds": fact_refunds,
    "fact_sessions": fact_sessions,
    "fact_pageviews": fact_pageviews,
}

In [ ]:
import pandas as pd
from google.cloud import bigquery

# Thông tin đích
project_id = "jda-k1"
dataset_id = "bigproject_phap_mart"
dataset = client.get_dataset(f"{project_id}.{dataset_id}")

upload_log = []

for table_id, df in fact_dim_tables.items():
    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    # Tránh ghi đè bảng bằng DataFrame rỗng ngoài ý muốn
    if df.empty:
        raise ValueError(f"{table_id} đang rỗng. Kiểm tra lại cleaning.")

    # Cấu hình nạp
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    # Nạp DataFrame đã clean
    job = client.load_table_from_dataframe(
        df,
        table_ref,
        job_config=job_config,
        location=dataset.location
    )
    job.result()

    # Đối chiếu số dòng
    loaded_table = client.get_table(table_ref)

    if loaded_table.num_rows != len(df):
        raise ValueError(f"Số dòng không khớp ở bảng {table_id}")

    upload_log.append({
        "table": table_id,
        "dataframe_rows": len(df),
        "bigquery_rows": loaded_table.num_rows,
        "status": "OK"
    })

    print(f"OK — {table_ref}: {loaded_table.num_rows:,} dòng")

display(pd.DataFrame(upload_log))

OK — jda-k1.bigproject_phap_mart.dim_products: 4 dòng
OK — jda-k1.bigproject_phap_mart.fact_orders: 32,313 dòng
OK — jda-k1.bigproject_phap_mart.fact_order_items: 40,025 dòng
OK — jda-k1.bigproject_phap_mart.fact_refunds: 1,731 dòng
OK — jda-k1.bigproject_phap_mart.fact_sessions: 472,871 dòng
OK — jda-k1.bigproject_phap_mart.fact_pageviews: 1,188,124 dòng


,table,dataframe_rows,bigquery_rows,status
0,dim_products,4,4,OK
1,fact_orders,32313,32313,OK
2,fact_order_items,40025,40025,OK
3,fact_refunds,1731,1731,OK
4,fact_sessions,472871,472871,OK
5,fact_pageviews,1188124,1188124,OK
